# 02 Define HWW likelihood (replacing repo Hbb-like chi2 input)

## Notebook purpose
Define the H→WW measurement likelihood used in EFT fits.

## Mathematical model
Given measured signal strength μ_obs with asymmetric uncertainties (+σ_up, -σ_dn), define:

χ²(μ_pred) = ((μ_pred-μ_obs)/σ)^2, where σ=σ_up if μ_pred≥μ_obs else σ_dn.

This is a piecewise Gaussian approximation for asymmetric errors.

## Interface to fit engine
The conversion `μ_pred = σ_pred/σ_SM` links EFT-predicted cross section to the measured observable.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from coffea import util

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p=Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p/marker).exists(): return p
        if p.parent==p: break
        p=p.parent
    raise FileNotFoundError(f'Could not find {marker} from {Path.cwd()}')

REPO=find_repo_root()
NBDIR=REPO/'notebooks_hww_fit'
LISTING=REPO/'smeft_contents.txt'
for p in ['/uscms_data/d3/azhou/smeft/analysis','/uscms_data/d3/azhou/smeft/analysis/hbb-coffea']:
    if p not in sys.path: sys.path.append(p)
import stxs_functions as sf
print('repo:',REPO)
mu_obs=0.45; sig_up=0.96; sig_dn=0.78
print('HWW μ =',mu_obs,'+',sig_up,'-',sig_dn)

In [ ]:
def chi2_hww(mu_pred,mu_obs=mu_obs,sig_up=sig_up,sig_dn=sig_dn):
    d=mu_pred-mu_obs
    s=sig_up if d>=0 else sig_dn
    return (d/s)**2

def mu_from_xsec(sig_pred_fb,sig_sm_fb):
    return sig_pred_fb/sig_sm_fb

In [ ]:
cfg={'mu_obs':mu_obs,'sig_up':sig_up,'sig_dn':sig_dn,'selection':'boosted VBF HWW, m_jj>1TeV'}
(NBDIR/'hww_measurement.json').write_text(json.dumps(cfg,indent=2))